# 1. Preprocesamiento de datos — Deteccion de fraude (PaySim)

Carga el dataset crudo `PS_20174392719_1491204439457_log.csv`, aplica feature
engineering y genera el split temporal pedido por el proyecto:

- **Referencia / entrenamiento**: `step <= 400`
- **Produccion (simulacion online)**: `step > 400`

Ver diccionario de datos completo en [`data/README.md`](../data/README.md)
y en el `README.md` principal del repo.

In [1]:
import gc
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import (
    DROP_COLS,
    PRODUCTION_DATA_PATH,
    PROCESSED_DIR,
    RAW_DATA_PATH,
    REFERENCE_DATA_PATH,
    REFERENCE_MAX_STEP,
)
from src.features import build_dummies, engineer_features

pd.set_option("display.max_columns", None)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Leyendo raw data desde:", RAW_DATA_PATH)

Leyendo raw data desde: D:\DataScienceResearchPeru\Machine_Learning_Engineering_2026_cohort_now\Proyecto_ML_Engineering_DSRP_III_2026_Fraud_Detection\data\raw\PS_20174392719_1491204439457_log.csv


In [2]:
# dtypes explicitos: sobre el dataset completo (~6.4M filas) el default de
# pandas (float64/int64 + object para strings) puede agotar la memoria
# disponible en equipos modestos. float32 alcanza de sobra para montos y
# saldos de este dataset; `type` tiene solo 5 valores -> category.
RAW_DTYPES = {
    "step": "int32",
    "type": "category",
    "amount": "float32",
    "nameOrig": "string",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "nameDest": "string",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}

df = pd.read_csv(RAW_DATA_PATH, dtype=RAW_DTYPES)
print(df.shape)
print(f"Memoria en uso: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

(6362620, 11)


Memoria en uso: 928.7 MB


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.639648,C1231006815,170136.0,160296.359375,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,C1666544295,21249.0,19384.720703,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.000000,C1305486145,181.0,0.000000,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.000000,C840083671,181.0,0.000000,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.139648,C2048537720,41554.0,29885.859375,M1230701703,0.0,0.0,0,0


## 1.1 Exploracion rapida

In [3]:
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            int32   
 1   type            category
 2   amount          float32 
 3   nameOrig        string  
 4   oldbalanceOrg   float32 
 5   newbalanceOrig  float32 
 6   nameDest        string  
 7   oldbalanceDest  float32 
 8   newbalanceDest  float32 
 9   isFraud         int8    
 10  isFlaggedFraud  int8    
dtypes: category(1), float32(5), int32(1), int8(2), string(2)
memory usage: 260.9 MB


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [4]:
print("Distribucion de la clase objetivo (isFraud):")
print(df["isFraud"].value_counts(normalize=True))
print()
print("Distribucion por tipo de transaccion:")
print(df["type"].value_counts())
print()
print("Tasa de fraude por tipo de transaccion:")
print(df.groupby("type")["isFraud"].mean().sort_values(ascending=False))

Distribucion de la clase objetivo (isFraud):
isFraud
0    0.998709
1    0.001291
Name: proportion, dtype: float64

Distribucion por tipo de transaccion:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Tasa de fraude por tipo de transaccion:


type
TRANSFER    0.007688
CASH_OUT    0.001840
CASH_IN     0.000000
DEBIT       0.000000
PAYMENT     0.000000
Name: isFraud, dtype: float64


C:\Users\HP\AppData\Local\Temp\ipykernel_15700\3854433658.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby("type")["isFraud"].mean().sort_values(ascending=False))


Nota de negocio (Anexo 1): el fraude en PaySim solo ocurre en
transacciones `TRANSFER` y `CASH_OUT` (el agente toma control de la cuenta,
transfiere el dinero y luego lo retira). Lo validamos a continuacion.

In [5]:
fraud_by_type = df.loc[df["isFraud"] == 1, "type"].value_counts()
print(fraud_by_type)
# Nota: con `type` como dtype category, value_counts() incluye categorias con
# conteo 0 (CASH_IN, DEBIT, PAYMENT); filtramos a las que de verdad tienen
# fraude antes de validar la regla de negocio del Anexo 1.
tipos_con_fraude = set(fraud_by_type[fraud_by_type > 0].index)
assert tipos_con_fraude <= {"TRANSFER", "CASH_OUT"}, "Fraude fuera de TRANSFER/CASH_OUT"


type
CASH_OUT    4116
TRANSFER    4097
CASH_IN        0
DEBIT          0
PAYMENT        0
Name: count, dtype: int64


## 1.2 Feature engineering

Ver `src/features.py` para el detalle de cada feature derivada.

In [6]:
df_feat = engineer_features(df)
del df  # ya no se necesita el crudo; libera ~1-2 GB antes del split/guardado
gc.collect()

df_feat = build_dummies(df_feat)

# nameOrig/nameDest (alta cardinalidad) e isFlaggedFraud ya cumplieron su
# proposito (derivar isDestMerchant / quedar fuera de los features del
# modelo, ver data/README.md) -- se sueltan para no cargar el resto del
# pipeline con columnas de texto pesadas que no se usan.
df_feat = df_feat.drop(columns=DROP_COLS)
gc.collect()

print(f"Memoria en uso (df_feat): {df_feat.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_feat.filter(regex="error|Ratio|empt|Merchant|Unknown|^type_").head()

Memoria en uso (df_feat): 642.6 MB


,errorBalanceOrig,errorBalanceDest,amountToOldBalanceOrgRatio,isDestMerchant,destBalanceUnknown,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,0.0,9839.639648,0.057834,1,1,0,0,0,1,0
1,0.0,1864.280029,0.087735,1,1,0,0,0,1,0
2,0.0,181.000000,1.000000,0,0,0,0,0,0,1
3,0.0,21363.000000,1.000000,0,0,0,1,0,0,0
4,0.0,11668.139648,0.280795,1,1,0,0,0,1,0


## 1.3 Split temporal: referencia vs produccion

`step <= 400` -> referencia (entrenamiento/validacion offline). `step > 400` -> produccion (simulacion de monitoreo online).

In [7]:
reference = df_feat[df_feat["step"] <= REFERENCE_MAX_STEP].reset_index(drop=True)
production = df_feat[df_feat["step"] > REFERENCE_MAX_STEP].reset_index(drop=True)

print(f"Referencia : {reference.shape} | tasa de fraude = {reference['isFraud'].mean():.5%}")
print(f"Produccion : {production.shape} | tasa de fraude = {production['isFraud'].mean():.5%}")

Referencia : (5787030, 18) | tasa de fraude = 0.07736%
Produccion : (575590, 18) | tasa de fraude = 0.64907%


In [8]:
reference.to_csv(REFERENCE_DATA_PATH, index=False)
production.to_csv(PRODUCTION_DATA_PATH, index=False)
print("Guardado:")
print(" -", REFERENCE_DATA_PATH)
print(" -", PRODUCTION_DATA_PATH)

Guardado:
 - D:\DataScienceResearchPeru\Machine_Learning_Engineering_2026_cohort_now\Proyecto_ML_Engineering_DSRP_III_2026_Fraud_Detection\data\processed\reference.csv
 - D:\DataScienceResearchPeru\Machine_Learning_Engineering_2026_cohort_now\Proyecto_ML_Engineering_DSRP_III_2026_Fraud_Detection\data\processed\production.csv


## 1.4 Siguiente paso

Continuar con [`2.0-entrenamiento-modelo.ipynb`](2.0-entrenamiento-modelo.ipynb)
para el entrenamiento, optimizacion de hiperparametros y registro en MLflow.